# 6.3 · 层次聚类 / Hierarchical Clustering

> **课程定位 / Where this fits**
> 第 3 课，**Part 6 · 无监督学习**。
> Lesson 3, **Part 6 · Unsupervised Learning**.
>
> K-Means(6.1)要预先指定簇数 K，而且只给出一个"扁平"的划分。层次聚类**不需要预先定 K**：它自底向上不断把最近的簇合并起来，生成一棵**树状图(dendrogram)**，想要几个簇就在合适的高度"横切一刀"。它还能揭示嵌套结构（大群里套着小群）。
> K-Means (6.1) needs a preset cluster count K and gives a single "flat" partition. Hierarchical clustering **needs no preset K**: it repeatedly merges the nearest clusters bottom-up, producing a **dendrogram**; cut it at any height for K clusters. It also reveals nested structure (groups within groups).

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - linkage —— 簇与簇之间距离的定义方式 / how distance between two clusters is defined
> - 树状图 dendrogram —— 记录合并顺序与高度的树 / a tree recording the merge order and heights
> - 合并高度 merge height —— 两个簇合并时的距离(越高越不相似) / the distance at which two clusters merge

> 💡 **面试相关 / Interview-relevant**
> - "凝聚 vs 分裂（agglomerative vs divisive）"（★★★★）
> - "linkage 准则：single/complete/average/ward 区别"（★★★★★）
> - "树状图怎么读 / 怎么定 K"（★★★★）
> - "层次聚类复杂度为什么是 O(n²~n³)"（★★★★，不适合大数据）

---

## 学习目标 / Learning Objectives

1. 理解凝聚式层次聚类流程 + 树状图。
   Understand agglomerative clustering and the dendrogram.
2. 区分四种 **linkage** 准则及其几何后果。
   Distinguish the four **linkage** criteria and their geometric effects.
3. 从树状图切割定 K。
   Cut the dendrogram to choose K.
4. 理解复杂度与适用边界。
   Understand its complexity and where it fits.

## 目录 / TOC
1. [先建直觉：不断合并最近的两团](#1)
2. [凝聚聚类与 linkage ⭐](#2)
3. [🛍️ 数据 + 树状图 ⭐](#3)
4. [linkage 准则对比 ⭐](#4)
5. [切树定 K + 对照 K-Means](#5)
6. [小结](#6)


<a id="1"></a>
## 1. 先建直觉：不断合并最近的两团 / Intuition First

想象每个数据点一开始都是一个"独立小团"。层次聚类的做法很像"滚雪球"：**反复找出当前最相近的两团，把它们合并成一团**，重复到最后只剩一团。
Imagine each data point starts as its own tiny cluster. Hierarchical clustering is like a snowball: **repeatedly find the two most similar clusters and merge them**, until only one remains.

把每次合并画下来，就得到一棵**树状图**：底部是单个点，越往上合并的团越大。两支在**很高处**才合并，说明它们很不相似。想分成几个簇？就在某个高度**横切一刀**，被切断的竖线条数就是簇数——所以 K 不用事先定，看树自己决定。
Recording every merge yields a **dendrogram**: single points at the bottom, larger clusters higher up. Two branches that only merge **high up** are very dissimilar. Want K clusters? **Cut horizontally** at some height; the number of vertical lines you cut = the number of clusters — so K need not be set in advance.


<a id="2"></a>
## 2. 凝聚聚类与 linkage ⭐ / Agglomerative & Linkage

**凝聚式(agglomerative，自底向上)**：每个点先各自成簇 → 反复**合并距离最近的两个簇** → 直到只剩一个。整个过程记录成一棵树。（**分裂式 divisive** 反过来自顶向下，少用。）
**Agglomerative (bottom-up):** each point starts as a cluster → repeatedly **merge the two nearest clusters** → until one remains. The process is recorded as a tree. (**Divisive** goes top-down; rarely used.)

关键问题是"两个**簇**之间的距离"怎么定义 —— 这就是 **linkage 准则**：
The key question is how to define the distance between two **clusters** — the **linkage** criterion:

| linkage | 簇间距离定义 / cluster distance | 倾向 / tendency |
|---|---|---|
| **single 单链** | 两簇**最近**的一对点的距离 / nearest pair | 链式，能捕捉细长/非球形，但易"链桥" |
| **complete 全链** | 两簇**最远**的一对点的距离 / farthest pair | 紧凑球形簇，对异常敏感 |
| **average 平均** | 所有跨簇点对的平均距离 / mean over all pairs | 折中 |
| **ward** | 合并后**簇内方差增量**最小 / min variance increase | 等大球形簇，最常用（类似 K-Means 目标）|

ward 与 K-Means 精神一致（都在最小化簇内方差），是默认首选。
ward shares K-Means's spirit (both minimize within-cluster variance) and is the default choice.


<a id="3"></a>
## 3. 数据 + 树状图 ⭐ / Data & Dendrogram

复用 6.1 的 **Mall Customers**（同一生成器），在 income/spending 两维上聚类并画树状图。
Reusing **Mall Customers** from 6.1 (same generator); cluster on income/spending and plot the dendrogram.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.preprocessing import StandardScaler
sns.set_theme(style="whitegrid")

def make_mall(seed=0):
    rng = np.random.default_rng(seed)
    groups = [((55,50),120,(25,60)),((25,80),35,(18,35)),((85,82),40,(28,42)),
              ((85,18),38,(35,60)),((26,18),35,(40,68))]
    rows=[]
    for (inc,spd),n,(amin,amax) in groups:
        rows.append(np.c_[rng.integers(amin,amax,n), rng.normal(inc,8,n).clip(15,140), rng.normal(spd,9,n).clip(1,99)])
    X=np.vstack(rows); rng.shuffle(X)
    return pd.DataFrame(X, columns=["age","income_k","spending"])

mall = make_mall()
print(f"Mall Customers: {mall.shape}")
Xs = StandardScaler().fit_transform(mall[["income_k","spending"]])   # 聚类前标准化(同 6.1)

# linkage 做凝聚聚类, 返回合并记录矩阵 Z(每行=一次合并: 哪两簇、合并距离、新簇大小)
Z = linkage(Xs, method="ward")
fig, ax = plt.subplots(figsize=(11, 4.5))
# dendrogram 把 Z 画成树; truncate_mode='lastp', p=30 表示只画最后 30 次合并(否则太密)
dendrogram(Z, truncate_mode="lastp", p=30, ax=ax)
ax.axhline(10, color="r", ls="--", label="切割线 cut → 5 簇")   # 在高度10横切, 得到5个簇
ax.set_xlabel("样本(截断显示) samples (truncated)"); ax.set_ylabel("合并距离(ward) merge distance"); ax.legend()
ax.set_title("树状图: 自底向上合并; 高度=合并代价; 横切一刀决定簇数 / cut to set #clusters")
plt.tight_layout(); plt.show()
print("读法: 越晚合并(越高)的两支越不相似; 在大间隙处切, 得到自然簇数")
print("Reading: branches merging higher are more dissimilar; cut at a big gap for natural clusters.")


<a id="4"></a>
## 4. linkage 准则对比 ⭐ / Linkage Comparison

不同 linkage 在同一份数据上会给出**完全不同**的簇形状。用月牙数据最能看出差异：single 能顺着形状捕捉弯曲的簇，complete/ward 则偏向球形。
Different linkages give **very different** cluster shapes on the same data. Moons show this best: single follows the shape and captures curved clusters, while complete/ward prefer spherical ones.


In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import make_moons
Xm, _ = make_moons(300, noise=0.06, random_state=0)

fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, lk in zip(axes, ["single","complete","average","ward"]):
    # AgglomerativeClustering 直接给簇标签; n_clusters=2 表示切成 2 簇
    lab = AgglomerativeClustering(n_clusters=2, linkage=lk).fit_predict(Xm)
    ax.scatter(Xm[:,0], Xm[:,1], c=lab, cmap="coolwarm", s=12)
    ax.set_title(f"linkage={lk}"); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("月牙数据: single 捕捉弯曲簇; complete/ward 偏向球形(切错) / single follows shape")
plt.tight_layout(); plt.show()
print("single: 抓住两条月牙(链式); complete/average/ward: 球形假设→切错")
print("→ 非球形/细长簇用 single; 紧凑等大簇用 ward / single for elongated, ward for compact")


<a id="5"></a>
## 5. 切树定 K + 对照 K-Means / Cutting & vs K-Means

`fcluster` 按"想要的簇数"或"距离阈值"把树切开。下面在 Mall 上切成 5 簇，和 K-Means(6.1) 对比。
`fcluster` cuts the tree by a desired cluster count or distance threshold. Below we cut Mall into 5 clusters and compare with K-Means (6.1).


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

hier_labels = fcluster(Z, t=5, criterion="maxclust")        # 从树里切出恰好 5 个簇
km_labels = KMeans(5, n_init=10, random_state=0).fit_predict(Xs)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
# silhouette_score: 轮廓系数(6.1 学过), 衡量聚类质量, 越高越好
axes[0].scatter(Xs[:,0], Xs[:,1], c=hier_labels, cmap="tab10", s=18)
axes[0].set_title(f"层次(ward, 切5簇) hierarchical silhouette={silhouette_score(Xs,hier_labels):.3f}")
axes[1].scatter(Xs[:,0], Xs[:,1], c=km_labels, cmap="tab10", s=18)
axes[1].set_title(f"K-Means(K=5) silhouette={silhouette_score(Xs,km_labels):.3f}")
for a in axes: a.set_xlabel("income (std)"); a.set_ylabel("spending (std)")
plt.tight_layout(); plt.show()
# ARI: 两种聚类划分的一致性(1=完全一致) / Adjusted Rand Index between the two partitions
print(f"两者划分高度一致 agreement ARI = {adjusted_rand_score(hier_labels, km_labels):.3f}")
print("球形簇上层次(ward)与 K-Means 结果接近; 层次的优势是不预设K + 树状结构可解释")
print("On spherical clusters ward ≈ K-Means; hierarchical's edge: no preset K + interpretable tree.")


<a id="6"></a>
## 6. 小结 / Summary

```
凝聚式: 每点成簇→反复合并最近两簇→树状图; 不预设K, 切树得任意簇数
linkage: single(最近点对, 链式/非球形) complete(最远, 紧凑) average(折中) ward(方差增量, 默认)
树状图: 高度=合并代价; 在大间隙横切定K
复杂度 O(n²)~O(n³) → 不适合大数据(用 K-Means/Mini-batch)
ward 与 K-Means 同精神(最小化簇内方差), 球形簇上结果相近
```

### 💡 面试速查 / Interview cheat-sheet
1. **凝聚（自底向上合并）vs 分裂（自顶向下）**。
   Agglomerative (bottom-up merging) vs divisive (top-down splitting).
2. **linkage**：single=最近点对（链式），complete=最远，ward=方差增量（默认/最常用）。
   single=nearest pair (chaining), complete=farthest, ward=variance increase (default).
3. **树状图**在大间隙处切定 K；不需预设 K 是核心优势。
   Cut the dendrogram at a big gap; no preset K is the key advantage.
4. **复杂度 O(n²~n³)** → 大数据不适用。
   O(n²~n³) → not for big data.
5. **vs K-Means**：层次不预设 K + 有层级结构；K-Means 快、可扩展。
   vs K-Means: hierarchical needs no K + has hierarchy; K-Means is fast and scalable.

### 下一节 / Next
**6.4 DBSCAN**——前面都假设簇是团状。DBSCAN 基于**密度**，能找任意形状的簇、自动识别噪声、且不需预设簇数。
**6.4 DBSCAN** — these assumed blob clusters. DBSCAN is **density-based**: arbitrary shapes, automatic noise detection, and no preset cluster count.
